In [ ]:
import scienceplots  # noqa: F401
import shutup
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2

# pretty plots
plt.style.use(["nature"])
plt.rcParams["figure.dpi"] = 200
%matplotlib widget
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}

# suppress warnings :-)
shutup.please()

In [ ]:
subj_id = "MR82"
sess_id = "20251027_152036"

## both

In [ ]:
from sg.models import Encoder

encoder = Encoder(subj_id, sess_id, norm=False)
encoder.fit_encoder()
encoder.encoder_predict()

In [ ]:
encoder.verify()

In [ ]:
# sanity check fits against scatter (unit 30)
# tv vs tv + drift
encoder.fit_baseline()
encoder.baseline_predict()

In [ ]:
from squiggs.renderers import FitRenderer
from squiggs.neuron_viewer import NeuronViewer
from utils.paths import FIGURES_DIR

reg = "DLS"

r = FitRenderer(
    y=encoder.robs[:, encoder.reg_idxs[reg]],
    yhat=encoder.robs_predict["baseline"][:, encoder.reg_idxs[reg]],
    mode="lite",
)

_ = NeuronViewer(
    num_units=encoder.psths[reg].shape[0], render_func=r, fig_dir=FIGURES_DIR
)

In [ ]:
from squiggs.renderers import PETHWeightRenderer
from squiggs.neuron_viewer import NeuronViewer
from utils.paths import FIGURES_DIR
from core.data import get_psths_cond, get_choice_ts, get_tavg_sc_cond

"""
drift: 21, 29, 35, 36
response: 16, 25, 26, 28
"""

reg = "DLS"
mode = "response"

sc_tavg = get_tavg_sc_cond(
    encoder.robs[:, encoder.reg_idxs[reg]], encoder.trial_data, cond=mode
)

r = PETHWeightRenderer(
    weights=encoder.encoder.coef_[encoder.reg_idxs[reg], :],
    weight_names=encoder.dm_names,
    robs=encoder.robs[:, encoder.reg_idxs[reg]],
    sc_tavg=sc_tavg,
    event_times=get_choice_ts(encoder.trial_data, mode=mode),
    spike_times=encoder.spike_times[reg],
    peths=get_psths_cond(encoder.psths[reg], encoder.trial_data, mode=mode),
    pres=0.5,
    posts=1,
    binwidth_s=25 / 1000,
)

nv = NeuronViewer(num_units=len(encoder.psths[reg]), render_func=r, fig_dir=FIGURES_DIR)

In [ ]:
from sg.models import ShuffledEncoder

se = ShuffledEncoder(
    subj_id,
    sess_id,
    tv_keys=[
        "response",
        "rewarded",
        "block_side",
        "strategy",
        "response_prev",
        "rewarded_prev",
    ],
)
se.plot_cvr2()
se.plot_dr2()

# strategy split

## single session

In [ ]:
from sg.models import StrategyEncoder

encoder_mb = StrategyEncoder(subj_id, sess_id, norm=False, strategy_filter="mb")
encoder_mb.fit_encoder()
encoder_mb.encoder_predict()

encoder_mf = StrategyEncoder(subj_id, sess_id, norm=False, strategy_filter="mf")
encoder_mf.fit_encoder()
encoder_mf.encoder_predict()

In [ ]:
encoder_mb.verify()
encoder_mf.verify()

In [ ]:
# pull out neurons above unity, look at neurons below unity

In [ ]:
from squiggs.renderers import FitRenderer
from squiggs.neuron_viewer import NeuronViewer
from utils.paths import FIGURES_DIR

reg = "DLS"

encoder_ = encoder_mf

r = FitRenderer(
    y=encoder_.robs[:, encoder_.reg_idxs[reg]],
    yhat=encoder_.robs_predict["encoder"][:, encoder_.reg_idxs[reg]],
    mode="lite",
)

_ = NeuronViewer(
    num_units=encoder_.psths[reg].shape[0], render_func=r, fig_dir=FIGURES_DIR
)

In [ ]:
import numpy as np

reg = "DLS"
regr = "response"
val = "right"
regr_idx_mb = np.where(encoder_mb.dm_names == f"{regr}_{val}")[0][0]
regr_idx_mf = np.where(encoder_mf.dm_names == f"{regr}_{val}")[0][0]

coef_mb = encoder_mb.encoder_weights[encoder_mb.reg_idxs[reg], regr_idx_mb]
coef_mf = encoder_mf.encoder_weights[encoder_mf.reg_idxs[reg], regr_idx_mf]

coef_diff = coef_mb - coef_mf
idxs = np.flip(
    np.argsort(np.abs(coef_diff))  # / encoder.robs.mean(axis=0)[encoder.reg_idxs[reg]])
)

In [ ]:
from squiggs.renderers import PETHWeightCompRenderer
from squiggs.neuron_viewer import NeuronViewer
from core.data import get_tavg_sc_cond, get_choice_ts, get_psths_cond
from utils.paths import FIGURES_DIR

reg = "DLS"
mode = "response"

sc_tavg_mb = get_tavg_sc_cond(
    encoder_mb.robs[:, encoder_mb.reg_idxs[reg]], encoder_mb.trial_data, cond=mode
)

sc_tavg_mf = get_tavg_sc_cond(
    encoder_mf.robs[:, encoder_mf.reg_idxs[reg]], encoder_mf.trial_data, cond=mode
)

r = PETHWeightCompRenderer(
    weights={
        "mb": encoder_mb.encoder_weights[encoder_mb.reg_idxs[reg], :],
        "mf": encoder_mf.encoder_weights[encoder_mf.reg_idxs[reg], :],
    },
    weight_names=encoder_mb.dm_names,
    robs={
        "mb": encoder_mb.robs[:, encoder_mb.reg_idxs[reg]],
        "mf": encoder_mf.robs[:, encoder_mf.reg_idxs[reg]],
    },
    sc_tavgs={"mb": sc_tavg_mb, "mf": sc_tavg_mf},
    event_times={
        "mb": get_choice_ts(encoder_mb.trial_data, mode=mode),
        "mf": get_choice_ts(encoder_mf.trial_data, mode=mode),
    },
    spike_times=encoder_mb.spike_times[reg],  # == encoder_mf.spike_times
    peths={
        "mb": get_psths_cond(encoder_mb.psths[reg], encoder_mb.trial_data, mode=mode),
        "mf": get_psths_cond(encoder_mf.psths[reg], encoder_mf.trial_data, mode=mode),
    },
    same_ylim=True,
)

_ = NeuronViewer(encoder_mb.psths[reg].shape[0], r, fig_dir=FIGURES_DIR)